# 第8章 静学ゲームの推定：Bresnahan and Reiss (1991) 参入モデル

MRIスキャナーの導入に関する参入モデルを推定する。
Bresnahan and Reiss (1991) の手法を用い、参入閾値と競争効果を分析する。

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import norm
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings('ignore')

# 日本語フォント設定
for font_name in ['Hiragino Maru Gothic Pro', 'Hiragino Sans',
                   'IPAexGothic', 'Noto Sans CJK JP', 'Yu Gothic']:
    try:
        matplotlib.font_manager.findfont(font_name, fallback_to_default=False)
        plt.rcParams['font.family'] = font_name
        break
    except ValueError:
        continue

# パス設定
base_dir = Path('..')
output_dir = base_dir / 'output'
output_dir.mkdir(exist_ok=True)

print("Setup complete.")


Setup complete.


## データの読み込みと準備

In [2]:
# 病院レベルのデータを読み込み
data = pd.read_csv(base_dir / 'data' / 'MRIData_BR1991.csv', encoding='utf-8')

print(f"病院レベルのデータ: {len(data)} 行")
print(f"カラム: {list(data.columns)}")
data.head()


病院レベルのデータ: 8862 行
カラム: ['HospitalID', 'CityCode', 'Kyukyu', 'Kinou', 'Sien', 'Hyoka', 'DepNeurology', 'DepNeurosurgery', 'NumBeds', 'MRINumOwn', 'Tesla', 'Population', 'Menseki', 'PopDensity', 'TaxableIncome', 'MRIOwnDum']


,HospitalID,CityCode,Kyukyu,Kinou,Sien,Hyoka,DepNeurology,DepNeurosurgery,NumBeds,MRINumOwn,Tesla,Population,Menseki,PopDensity,TaxableIncome,MRIOwnDum
0,1,1101,1.0,0.0,0.0,0.0,0.0,0.0,243.0,1,2.0,220189.0,46.42,4743.4,3017.0,1
1,2,1101,0.0,0.0,0.0,0.0,0.0,0.0,180.0,1,0.0,220189.0,46.42,4743.4,3017.0,1
2,3,1101,1.0,0.0,0.0,0.0,0.0,0.0,110.0,1,1.0,220189.0,46.42,4743.4,3017.0,1
3,4,1101,1.0,0.0,0.0,0.0,1.0,1.0,134.0,3,2.0,220189.0,46.42,4743.4,3017.0,1
4,5,1101,0.0,0.0,0.0,0.0,0.0,0.0,250.0,1,2.0,220189.0,46.42,4743.4,3017.0,1


In [3]:
# 病院レベルのデータを市区町村レベルの集計データに変換
listCode = data['CityCode'].unique()

records = []
for code in listCode:
    subdata = data[data['CityCode'] == code]
    records.append({
        'Code': code,
        'NumHospital': len(subdata),
        'NumMRI': subdata['MRIOwnDum'].sum(),
        'Pop': subdata['Population'].unique()[0],
        'Menseki': subdata['Menseki'].unique()[0],
        'PopDen': subdata['PopDensity'].unique()[0],
        'Income': subdata['TaxableIncome'].unique()[0],
    })

dataset = pd.DataFrame(records)
print(f"市区町村数: {len(dataset)}")
dataset.head()


市区町村数: 1509


,Code,NumHospital,NumMRI,Pop,Menseki,PopDen,Income
0,1101,41,17,220189.0,46.42,4743.4,3017.0
1,1102,25,4,278781.0,63.48,4391.6,3017.0
2,1103,23,10,255873.0,57.13,4478.8,3017.0
3,1104,23,7,204259.0,34.58,5906.9,3017.0
4,1105,18,6,212118.0,46.35,4576.4,3017.0


## 表8.3: 病院数とMRI保有病院数のクロス集計

In [4]:
# 10以上は10としてまとめる
dt_table = dataset[['NumHospital', 'NumMRI']].copy()
dt_table['NumHospital'] = dt_table['NumHospital'].clip(upper=10)
dt_table['NumMRI'] = dt_table['NumMRI'].clip(upper=10)

# クロス集計
tbl = pd.crosstab(dt_table['NumHospital'], dt_table['NumMRI'])
print("病院数 x MRI保有病院数のクロス集計:")
print(tbl)

# CSV保存
tbl.to_csv(output_dir / 'Tab8_3_hospital_mri_table.csv')
print("\n保存: output/Tab8_3_hospital_mri_table.csv")


病院数 x MRI保有病院数のクロス集計:
NumMRI        0    1   2   3   4   5   6   7   8   9   10
NumHospital                                              
1            257  149   0   0   0   0   0   0   0   0   0
2             72  117  24   0   0   0   0   0   0   0   0
3             34   77  44  15   0   0   0   0   0   0   0
4             14   55  54  15   5   0   0   0   0   0   0
5              4   29  30  30   7   0   0   0   0   0   0
6              2   17  27  25   7   3   1   0   0   0   0
7              0   11  13  15  15   5   1   0   0   0   0
8              1    7  11  12   6   6   5   2   0   0   0
9              1    1  10   9   9   6   4   1   0   0   0
10             0    2  18  21  32  31  32  20  26  12  50

保存: output/Tab8_3_hospital_mri_table.csv


## 目的関数の定義

Bresnahan and Reiss (1991) モデルの対数尤度関数を定義する。

In [5]:
def obj(params, dataset_NumMRI, dataset_Pop, M, N_max):
    """
    Bresnahan-Reiss (1991) モデルの目的関数（正規化対数尤度）。

    Parameters
    ----------
    params : array, shape (N_max + 1,)
        パラメータ。params[0]=alpha_1, params[1:N_max]=-alpha_2,...,-alpha_{N_max}, params[N_max]=gamma
    dataset_NumMRI : array, shape (M,)
        各市区町村のMRI導入病院数
    dataset_Pop : array, shape (M,)
        各市区町村の人口（百万人単位）
    M : int
        市区町村数
    N_max : int
        最大参入企業数

    Returns
    -------
    float
        正規化対数尤度 (sum of log-likelihood / M)
    """
    # パラメータの定義
    alpha1 = params[0]
    alpha2 = -params[1:N_max]  # alpha_2, ..., alpha_{N_max} にはマイナスを乗じる
    alpha = np.concatenate([[alpha1], alpha2])
    gamma = params[N_max]

    # 人口の行列 (M x N_max)
    pop = np.tile(dataset_Pop.reshape(-1, 1), (1, N_max))

    # 下三角行列 V を構築
    # V[i:, i] = alpha[i] (i = 0, ..., len(alpha)-1)
    V = np.zeros((N_max, N_max))
    for i in range(len(alpha)):
        V[i:, i] = alpha[i]

    # VV: 可変利潤部分 (M x N_max)
    VV = (V @ np.ones((N_max, M))).T

    # 固定費用の行列 (M x N_max)
    F = np.full((M, N_max), gamma)

    # 利潤行列
    pi_mat = pop * VV - F

    # 標準正規分布のCDFを適用
    phi = norm.cdf(pi_mat)

    # 各企業数になる確率を計算
    # n=0: 1 - phi[:,0]
    # n=k (0 < k < N_max): phi[:,k-1] - phi[:,k]
    # n=N_max: phi[:,N_max-1]
    mat = np.column_stack([
        1 - phi[:, 0],
        phi[:, :N_max - 1] - phi[:, 1:N_max],
        phi[:, N_max - 1]
    ])

    # 観測された企業数に対応する確率の対数を抽出
    ml = np.zeros(M)
    for i in range(N_max + 1):
        mask = dataset_NumMRI == i
        ml[mask] = np.log(np.maximum(mat[mask, i], 1e-300))

    # 対数尤度が定義できない場合の対処
    ml[np.isinf(ml)] = -10000

    # 正規化対数尤度
    val = np.sum(ml) / M
    return val


print("目的関数の定義完了")


目的関数の定義完了


## 数値ヘシアンの関数定義

In [6]:
def numerical_hessian(func, x, eps=1e-5):
    """
    有限差分法による数値ヘシアンの計算。

    Parameters
    ----------
    func : callable
        スカラー値を返す関数
    x : array
        パラメータの値
    eps : float
        差分の幅

    Returns
    -------
    H : array, shape (n, n)
        ヘシアン行列
    """
    n = len(x)
    H = np.zeros((n, n))
    f0 = func(x)
    for i in range(n):
        for j in range(n):
            e_i = np.zeros(n)
            e_j = np.zeros(n)
            e_i[i] = eps
            e_j[j] = eps
            H[i, j] = (func(x + e_i + e_j)
                        - func(x + e_i)
                        - func(x + e_j)
                        + f0) / eps**2
    return H


print("数値ヘシアン関数の定義完了")


数値ヘシアン関数の定義完了


## Bresnahan and Reiss (1991) モデルの推定

N_max = 6, 7, 8, 9, 10 について推定を行い、参入閾値を計算する。

In [7]:
# 変数に欠損がある市区町村を落とす
dataset_clean = dataset.dropna().copy()

# 人口を百万で除する
dataset_clean['Pop'] = dataset_clean['Pop'] / 1_000_000

# 推定で用いるサンプルサイズ（市区町村数）
M = len(dataset_clean)
print(f"推定サンプルサイズ（市区町村数）: {M}")

# オリジナルのデータセットを保持
dataset_org = dataset_clean.copy()

# N_maxを6から10まで試す
N_max_list = list(range(6, 11))

# 結果の保管場所
result_estimates = {}
result_thresholds = {}

for N_max in N_max_list:
    print(f"\n{'='*60}")
    print(f"N_max: {N_max}")
    print(f"{'='*60}")

    # データセットを初期化
    ds = dataset_org.copy()

    # MRI導入病院数がN_maxより大きい場合、N_maxに置換
    ds.loc[ds['NumMRI'] > N_max, 'NumMRI'] = N_max

    # 推定に使う変数を抽出
    dataset_NumMRI = ds['NumMRI'].values.astype(float)
    dataset_Pop = ds['Pop'].values.astype(float)

    # パラメータの初期値を全て1に設定
    initial = np.ones(N_max + 1)

    # 最適化（最大化するので目的関数を符号反転）
    result = minimize(
        lambda p: -obj(p, dataset_NumMRI, dataset_Pop, M, N_max),
        initial,
        method='L-BFGS-B',
        bounds=[(0, None)] * (N_max + 1)
    )

    estimates = result.x
    print(f"最適化成功: {result.success}")
    print(f"目的関数値（正規化対数尤度）: {-result.fun:.6f}")

    # ヘシアンの計算
    def neg_obj_for_hessian(p):
        return -obj(p, dataset_NumMRI, dataset_Pop, M, N_max)

    H = numerical_hessian(neg_obj_for_hessian, estimates)

    # 標準誤差の計算
    try:
        se = np.sqrt(np.diag(np.linalg.inv(H) / M))
    except np.linalg.LinAlgError:
        print("Warning: ヘシアンの逆行列が計算できません")
        se = np.full(N_max + 1, np.nan)

    # 推定値と標準誤差を表示
    print("\n推定結果:")
    param_names = [f"alpha_1"] + [f"-alpha_{i}" for i in range(2, N_max + 1)] + ["gamma"]
    for name, est, s in zip(param_names, estimates, se):
        print(f"  {name:>12s}: {est:12.6f}  (se: {s:.8f})")

    # 結果を保管
    result_estimates[f"N_max={N_max}"] = np.column_stack([estimates, se])

    # === Entry Threshold の計算 ===
    alpha = estimates[:N_max]
    # alpha[0] = alpha_1, estimates[1:N_max] に -1 を乗じたものが alpha_2,...
    # ただし obj 内では alpha = [alpha1, -params[1:N_max]] としているので
    # 実際の alpha ベクトルは alpha_1, -estimates[1], -estimates[2], ...
    alpha_vals = np.zeros(N_max)
    alpha_vals[0] = estimates[0]
    alpha_vals[1:] = -estimates[1:N_max]

    gamma_val = estimates[N_max]

    EntryThreshold = np.zeros((N_max, 3))  # S_N, s_N, ratio

    # S_1, s_1
    deno = alpha_vals[0]
    S_N = gamma_val / deno * 1e6
    EntryThreshold[0, 0] = int(S_N)
    EntryThreshold[0, 1] = int(S_N)

    # S_n, s_n (n >= 2)
    for i in range(1, N_max):
        deno = deno + alpha_vals[i]  # alpha_vals[i] は負なので足す = 引く
        S_N = gamma_val / deno * 1e6
        EntryThreshold[i, 0] = int(S_N)
        EntryThreshold[i, 1] = int(S_N / (i + 1))

    # 比率 s_{N+1}/s_N
    for j in range(N_max):
        if j < N_max - 1:
            EntryThreshold[j, 2] = EntryThreshold[j + 1, 1] / EntryThreshold[j, 1]
        else:
            EntryThreshold[j, 2] = np.nan

    print("\nEntry Threshold:")
    print(f"  {'N':>3s}  {'S_N':>10s}  {'s_N=S_N/N':>12s}  {'s_{N+1}/s_N':>14s}")
    for i in range(N_max):
        ratio_str = f"{EntryThreshold[i, 2]:.6f}" if not np.isnan(EntryThreshold[i, 2]) else "NA"
        print(f"  {i+1:>3d}  {EntryThreshold[i, 0]:>10.0f}  {EntryThreshold[i, 1]:>12.0f}  {ratio_str:>14s}")

    result_thresholds[f"N_max={N_max}"] = EntryThreshold


推定サンプルサイズ（市区町村数）: 1459

N_max: 6
最適化成功: True
目的関数値（正規化対数尤度）: -1.496528

推定結果:
       alpha_1:    55.578623  (se: nan)
      -alpha_2:    33.290971  (se: 0.53529113)
      -alpha_3:     9.320431  (se: 0.52162552)
      -alpha_4:     4.226974  (se: 0.32233779)
      -alpha_5:     2.322138  (se: 0.24110679)
      -alpha_6:     1.794860  (se: 0.23168195)
         gamma:     1.404724  (se: 0.00370952)

Entry Threshold:
    N         S_N     s_N=S_N/N     s_{N+1}/s_N
    1       25274         25274        1.246854
    2       63027         31513        1.145845
    3      108328         36109        1.112714
    4      160719         40179        1.089450
    5      218868         43773        1.156855
    6      303839         50639              NA

N_max: 7


最適化成功: True
目的関数値（正規化対数尤度）: -1.558291

推定結果:
       alpha_1:    55.630151  (se: nan)
      -alpha_2:    34.200960  (se: 0.57799863)
      -alpha_3:     8.112260  (se: 0.47394237)
      -alpha_4:     4.163665  (se: 0.31904401)
      -alpha_5:     2.447400  (se: 0.25299947)
      -alpha_6:     1.534422  (se: 0.20321836)
      -alpha_7:     1.440086  (se: 0.20473189)
         gamma:     1.424083  (se: 0.03617850)

Entry Threshold:
    N         S_N     s_N=S_N/N     s_{N+1}/s_N
    1       25599         25599        1.297980
    2       66455         33227        1.072772
    3      106937         35645        1.091177
    4      155581         38895        1.091966
    5      212363         42472        1.080594
    6      275374         45895        1.187951
    7      381652         54521              NA

N_max: 8


最適化成功: True
目的関数値（正規化対数尤度）: -1.598648

推定結果:
       alpha_1:    56.687595  (se: nan)
      -alpha_2:    34.926649  (se: 0.59588004)
      -alpha_3:     8.166219  (se: 0.47707845)
      -alpha_4:     4.232827  (se: 0.32358279)
      -alpha_5:     2.477931  (se: 0.25589331)
      -alpha_6:     1.574243  (se: 0.20789785)
      -alpha_7:     1.389865  (se: 0.19846765)
      -alpha_8:     0.814067  (se: 0.16238158)
         gamma:     1.445959  (se: 0.03580946)

Entry Threshold:
    N         S_N     s_N=S_N/N     s_{N+1}/s_N
    1       25507         25507        1.302505
    2       66447         33223        1.067122
    3      106361         35453        1.089104
    4      154451         38612        1.087978
    5      210047         42009        1.080411
    6      272322         45387        1.161059
    7      368880         52697        1.104351
    8      465568         58196              NA

N_max: 9


最適化成功: True
目的関数値（正規化対数尤度）: -1.637936

推定結果:
       alpha_1:    59.955946  (se: 0.00002183)
      -alpha_2:    37.629936  (se: 0.51758980)
      -alpha_3:     8.339902  (se: 0.48444488)
      -alpha_4:     4.294268  (se: 0.32603576)
      -alpha_5:     2.495326  (se: 0.25639006)
      -alpha_6:     1.589460  (se: 0.20912631)
      -alpha_7:     1.418344  (se: 0.20145434)
      -alpha_8:     0.811743  (se: 0.16176452)
      -alpha_9:     0.918204  (se: 0.17370208)
         gamma:     1.489963  (se: 0.00053850)

Entry Threshold:
    N         S_N     s_N=S_N/N     s_{N+1}/s_N
    1       24850         24850        1.342777
    2       66736         33368        1.064193
    3      106531         35510        1.082315
    4      153733         38433        1.077381
    5      207039         41407        1.069578
    6      265730         44288        1.147376
    7      355709         50815        1.085329
    8      441213         55151        1.220848
    9      605980         67331    

最適化成功: True
目的関数値（正規化対数尤度）: -1.658364

推定結果:
       alpha_1:    57.521373  (se: nan)
      -alpha_2:    35.403077  (se: 0.51199279)
      -alpha_3:     8.214217  (se: 0.47741843)
      -alpha_4:     4.232376  (se: 0.32202563)
      -alpha_5:     2.507382  (se: 0.25726065)
      -alpha_6:     1.552036  (se: 0.20471960)
      -alpha_7:     1.376797  (se: 0.19627180)
      -alpha_8:     0.795594  (se: 0.15883977)
      -alpha_9:     0.922575  (se: 0.17454822)
     -alpha_10:     0.493695  (se: 0.13771531)
         gamma:     1.477099  (se: nan)

Entry Threshold:
    N         S_N     s_N=S_N/N     s_{N+1}/s_N
    1       25679         25679        1.300284
    2       66781         33390        1.060527
    3      106234         35411        1.078196
    4      152723         38180        1.079990
    5      206174         41234        1.063807
    6      263190         43865        1.135757
    7      348743         49820        1.077379
    8      429402         53675        1.214662
  

## 結果の保存

In [8]:
# 推定値の保存
with open(output_dir / 'Tab8_4_BR1991_Estimates.txt', 'w') as f:
    for name, est in result_estimates.items():
        f.write(f"\n{name}\n")
        f.write(f"{'estimates':>14s} {'se':>14s}\n")
        for row in est:
            f.write(f"{row[0]:14.6f} {row[1]:14.8f}\n")

print("保存: output/Tab8_4_BR1991_Estimates.txt")

# Entry Threshold の保存
with open(output_dir / 'Tab8_5_Entry_Thresholds.txt', 'w') as f:
    for name, thr in result_thresholds.items():
        f.write(f"\n{name}\n")
        f.write(f"{'S_N':>10s} {'s_N=S_N/N':>12s} {'s_{N+1}/s_N':>14s}\n")
        for row in thr:
            ratio_str = f"{row[2]:14.6f}" if not np.isnan(row[2]) else f"{'NA':>14s}"
            f.write(f"{row[0]:10.0f} {row[1]:12.0f} {ratio_str}\n")

print("保存: output/Tab8_5_Entry_Thresholds.txt")


保存: output/Tab8_4_BR1991_Estimates.txt
保存: output/Tab8_5_Entry_Thresholds.txt


## 結果のまとめ

In [9]:
# 全てのN_maxの推定結果をまとめて表示
print("=" * 80)
print("Bresnahan and Reiss (1991) 推定結果のまとめ")
print("=" * 80)

for name in result_estimates:
    est = result_estimates[name]
    thr = result_thresholds[name]
    N_max = int(name.split("=")[1])

    print(f"\n--- {name} ---")
    print(f"  パラメータ推定値:")
    print(f"    {'param':>12s}  {'estimate':>12s}  {'se':>12s}")
    param_names = ["alpha_1"] + [f"-alpha_{i}" for i in range(2, N_max + 1)] + ["gamma"]
    for i, pname in enumerate(param_names):
        print(f"    {pname:>12s}  {est[i, 0]:12.6f}  {est[i, 1]:12.8f}")

    print(f"  Entry Threshold:")
    print(f"    {'N':>3s}  {'S_N':>10s}  {'s_N=S_N/N':>12s}  {'s_{N+1}/s_N':>14s}")
    for i in range(N_max):
        ratio_str = f"{thr[i, 2]:.6f}" if not np.isnan(thr[i, 2]) else "NA"
        print(f"    {i+1:>3d}  {thr[i, 0]:>10.0f}  {thr[i, 1]:>12.0f}  {ratio_str:>14s}")

print("\n" + "=" * 80)
print("推定完了")


Bresnahan and Reiss (1991) 推定結果のまとめ

--- N_max=6 ---
  パラメータ推定値:
           param      estimate            se
         alpha_1     55.578623           nan
        -alpha_2     33.290971    0.53529113
        -alpha_3      9.320431    0.52162552
        -alpha_4      4.226974    0.32233779
        -alpha_5      2.322138    0.24110679
        -alpha_6      1.794860    0.23168195
           gamma      1.404724    0.00370952
  Entry Threshold:
      N         S_N     s_N=S_N/N     s_{N+1}/s_N
      1       25274         25274        1.246854
      2       63027         31513        1.145845
      3      108328         36109        1.112714
      4      160719         40179        1.089450
      5      218868         43773        1.156855
      6      303839         50639              NA

--- N_max=7 ---
  パラメータ推定値:
           param      estimate            se
         alpha_1     55.630151           nan
        -alpha_2     34.200960    0.57799863
        -alpha_3      8.112260    0.473942